# A股 Level-2 首轮静态成本探索

三日、确定性证券子样本的描述性测量。动态冲击、队列位置和真实订单校准尚未完成。供应商聚合衍生数据的公开许可待确认，执行版与数值结果仅存本地。

## 方法与假设
价格以分存储、数量为股、时钟为 `HHMMSSmmm`。仅保留沪深股票代码前缀且数值代码模37为0的证券；连续竞价每证券每分钟选源文件中时间最早的一条，同时间按源行号打破平局。上午09:30至11:30之前，下午13:00至14:57之前。无PIT证券资格表，不包含北京市场，不能声称全市场代表性。

名义规模是中间价 × 分析订单股数，不是现金预算。允许分析用的非整手股数，不模拟实际整手委托。只对有效双边、有序盘口计算；单边盘口先计入剔除账本。超过十档深度的完整订单成本记缺失，不外推。价差已包含在中间价基准成本中，不重复计费。

In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
run_root = Path(os.environ['EXECUTION_COST_PILOT_RUN'])
summary = json.loads((run_root / 'summary.json').read_text())
audit = pd.DataFrame(summary['audit'])
audit[['day','rows_read','sampled_securities','sampled_snapshots','valid_books','excluded_books','day_mismatch','invalid_clock']]

## 独立复核
下面从落盘的证券分钟测量重新计算覆盖比例与分位数，避免图表数字仅依赖汇总JSON。结果是按快照计权的描述性分布，没有全市场置信区间，也没有未参与研究时期的样本外验证。

In [ ]:
measurements = pd.concat([pd.read_parquet(p) for p in sorted(run_root.glob('sampled_measurements_*.parquet'))], ignore_index=True)
assert len(measurements) == summary['pooled']['snapshots']
assert not measurements.duplicated(['TradingDay','SecuCode','minute']).any()
for row in summary['pooled']['cost_surface']:
    key = f"{row['side']}_{row['reference_notional_cny']}"
    cost = measurements[key + '_cost_bp'].dropna()
    assert np.isclose(len(cost) / len(measurements), row['full_coverage_rate'])
    if len(cost):
        assert np.isclose(cost.median(), row['full_order_cost_bp']['p50'])
    assert measurements[key + '_coverage'].between(0,1).all()
assert int(audit['rows_read'].sum()) == int(audit['metadata_rows'].sum())
assert audit['day_mismatch'].sum() == 0 and audit['invalid_clock'].sum() == 0
pd.DataFrame([{'side':r['side'],'reference_notional_cny':r['reference_notional_cny'],
               'fully_covered_n':r['full_order_cost_bp']['count'], 'coverage_pct':r['full_coverage_rate']*100,
               'conditional_p50_bp':r['full_order_cost_bp']['p50'], 'conditional_p90_bp':r['full_order_cost_bp']['p90'],
               'conditional_p95_bp':r['full_order_cost_bp']['p95']}
              for r in summary['pooled']['cost_surface']])

## 成本与覆盖率一起看
上图成本仅包含十档能完整覆盖的快照，因此不同规模的分位数来自不同样本。右下图限定为买卖所有规模都能完整覆盖的同一批快照，才适合比较订单规模本身。左下三日期柱图不是连续市场趋势。

In [ ]:
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
fig, axes = plt.subplots(2,2,figsize=(12,8),constrained_layout=True)
colors={'buy':'#b64d33','sell':'#206b80'}
for side in ['buy','sell']:
    rows = [r for r in summary['pooled']['cost_surface'] if r['side']==side]
    x=[r['reference_notional_cny'] for r in rows]
    axes[0,0].plot(x,[r['full_order_cost_bp']['p50'] for r in rows],marker='o',color=colors[side],label=side+' P50')
    axes[0,0].plot(x,[r['full_order_cost_bp']['p90'] for r in rows],marker='.',linestyle='--',color=colors[side],label=side+' P90')
    axes[0,1].plot(x,[100*r['full_coverage_rate'] for r in rows],marker='o',color=colors[side],label=side)
    matched=[r for r in summary['pooled']['common_full_subset']['cost_surface'] if r['side']==side]
    axes[1,1].plot(x,[r['full_order_cost_bp']['p50'] for r in matched],marker='o',color=colors[side],label=side+' P50')
for ax in [axes[0,0],axes[0,1],axes[1,1]]:
    ax.set_xscale('log'); ax.set_xticks(x,labels=['10k','100k','1m','10m']); ax.set_xlabel('Reference notional (CNY)')
    ax.legend(frameon=False); ax.grid(axis='y',alpha=.2); ax.set_ylim(bottom=0)
axes[0,0].set(title='Cost conditional on full ten-level coverage',ylabel='Cost versus midpoint (bp)')
axes[0,1].set(title='Full ten-level coverage among valid books',ylabel='Fully covered snapshots (%)',ylim=(0,105))
days=list(summary['by_day'])
axes[1,0].bar(days,[summary['by_day'][d]['spread_bp']['p50'] for d in days],color='#687f68')
axes[1,0].set(title='Median quoted spread by sampled day',ylabel='Full quoted spread (bp)',xlabel='Three selected dates (not a time series)')
axes[1,1].set(title=f"Same population: {summary['pooled']['common_full_subset']['snapshots']:,} snapshots",ylabel='Median cost versus midpoint (bp)')
fig.suptitle(f"Static L2 pilot · {len(measurements):,} valid stock-minute books · 3 days",fontweight='bold')
fig.savefig(run_root/'cost-and-coverage.png',dpi=160)
fig.savefig(run_root/'cost-and-coverage.svg')
plt.show()

## 限制与下一步
完整订单分位数会排除深度不足的快照，不能解释成平均实盘成交成本。先选每分钟第一条再做盘口过滤，避免在该分钟内寻找较好盘口。单边盘口仍可能存在一侧可交易流动性，本轮为共同买卖比较而保守排除。

按日中位十档金额分组只是同日描述性标签，使用当日完整数据，不能作为事前策略信号；不等于市值分组。资金规模不等于单笔订单规模。三日未做概率抽样，不构成全市场校准或样本外检验。

下一步：确认再分发条款，补充PIT证券资格、滞后ADV/市值、价格限制状态及更多预先选定日期，再考虑真实委托校准。未修复源事件排序，也未改变数据平台发布别名。

In [ ]:
# Private provenance: source hashes are preserved; do not publish source records.
print('Measurement code SHA256:',summary['measurement_code_sha256'])
print('Source input hashes:',[(r['day'],r['sha256']) for r in summary['inputs']])
print('Validation: independent aggregate checks passed')